[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbHansen/wb-debt-simulation/blob/main/optimization/kes/currency-MV-xlsx.ipynb)

# Currency MV Frontier -- KES Local Data Source

Loads KES FX rates from **`currency/kes_wide.xlsx`** (Central Bank of Kenya),
converts to quarterly data, explores the data visually, and runs a
mean-variance (MV) efficient frontier optimisation for the currency
composition of sovereign debt.

| | |
|---|---|
| **Data source** | `currency/kes_wide.xlsx` (daily, CBK rates) |
| **Currencies** | USD, GBP, EUR, JPY, CNY vs KES |
| **Resampling** | Quarterly mean (configurable) |

The pipeline is identical to `currency-MV.ipynb` except that the data
come from the local `kes_wide.xlsx` file via `get_kes_iso()` instead of
the ECB API.

In [ ]:
if 'google.colab' in str(get_ipython()):
    import os
    os.system('pip -qqq install ModelFlowIb openpyxl')
    os.system(
        'curl -sL -o exchangerates_get.py '
        'https://raw.githubusercontent.com/IbHansen/wb-debt-simulation/main/'
        'optimization/exchangerates_get.py'
    )
    os.system(
        'curl -sL -o get_kes.py '
        'https://raw.githubusercontent.com/IbHansen/wb-debt-simulation/main/'
        'optimization/kes/get_kes.py'
    )
    os.makedirs('currency', exist_ok=True)
    print("Please upload 'kes_wide.xlsx' when the file picker appears:")
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded:
        import shutil; shutil.move(fn, f'currency/{fn}')


In [ ]:
import sys
from pathlib import Path
# Make exchangerates_get (in the parent folder) importable when running locally
if str(Path('..').resolve()) not in sys.path:
    sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import numpy as np
import exchangerates_get as er
from get_kes import get_kes_iso


---
## Section 1 -- Loading and Exploring FX Data

`get_kes_iso()` reads `currency/kes_wide.xlsx`, inverts CBK rates to
KES-as-base convention (`KES_USD`, `KES_EUR`, ...), filters to the
requested currencies and resamples to the chosen frequency.

In [ ]:
# Load and resample KES FX rates
fx_ccy = get_kes_iso(
    currencies=['USD', 'GBP', 'EUR', 'JPY', 'CNY'],
    start='2016-01-01',
    freq='QE',
    agg='mean',
)
print(f'Period     : {fx_ccy.index[0]}  ->  {fx_ccy.index[-1]}  ({len(fx_ccy)} quarters)')
print(f'Currencies : {list(fx_ccy.columns)}')
fx_ccy


In [ ]:
er.plot_indexed_fx(
    fx_ccy,
    min_label_gap=5.0,
    title="FX indices vs KES (base = 100 at first observation)",
)

### 1.2 Quarterly Log-Returns and Annualised Volatility

Returns are computed as log-differences of the quarterly averages.

**Annualisation:** because interest rates in the optimiser are expressed as
*annual* rates, the risk axis must also be in annual units.  
The scaling rule for i.i.d. returns over T periods per year:

$$
\sigma_{\text{annual}} = \sqrt{T} \times \sigma_{\text{period}}
\qquad
\Sigma_{\text{annual}} = T \times \Sigma_{\text{period}}
$$

For **quarterly** data $T = 4$, so $\sigma_{\text{annual}} = 2 \times \sigma_{\text{quarterly}}$.

In [ ]:
import numpy as np

fx_returns = er.get_fx_returns(fx_ccy)
print(f"Return observations: {len(fx_returns)} quarters")
print()

display_stats = fx_returns.describe().T[['mean', 'std', 'min', 'max']].copy()
display_stats.columns = ['Mean (qtrly)', 'Std (qtrly)', 'Min', 'Max']

# annualise: quarterly std × sqrt(4) = quarterly std × 2
display_stats['Std % (qtrly)']  = fx_returns.std() * 100
display_stats['Std % (annual)'] = fx_returns.std() * np.sqrt(4) * 100   # ×2

print("Quarterly log-returns — std % quarterly vs annual (×2):")
display_stats.style.format({
    'Mean (qtrly)':   '{:.4f}',
    'Std (qtrly)':    '{:.4f}',
    'Min':            '{:.4f}',
    'Max':            '{:.4f}',
    'Std % (qtrly)':  '{:.2f}',
    'Std % (annual)': '{:.2f}',
})

### 1.3 Correlation Matrix (std % on diagonal)

In [ ]:
er.plot_corr_with_std(
    fx_returns,
    title="Quarterly log-return correlation (std on diagonal, %)"
)

### 1.4 Scatterplot Matrix

In [ ]:
er.plot_return_scatter_matrix_with_marginals(
    fx_returns,
    title="Quarterly log-return scatterplot matrix"
)

### 1.5 Covariance Matrix — Quarterly vs Annual

The covariance matrix fed into the optimiser must be in **annual** units so the
risk axis is comparable to the annual interest rates on the cost axis.

`get_fx_covariance(..., periods_per_year=4)` multiplies the raw quarterly matrix by 4:

| | Formula | Effect on std diagonal |
|---|---|---|
| Quarterly | $\Sigma_Q = \text{Cov}(r_Q)$ | $\sigma_Q$ |
| **Annual (used in MV)** | $\Sigma_A = 4 \times \Sigma_Q$ | $\sigma_A = 2 \times \sigma_Q$ |

In [ ]:
# Quarterly covariance (raw)
fx_cov_q = er.get_fx_covariance(fx_returns)

# Annual covariance: multiply by 4  → annual std = quarterly std × 2
fx_cov = er.get_fx_covariance(fx_returns, periods_per_year=4)

std_q = pd.Series(np.sqrt(np.diag(fx_cov_q.values)) * 100,
                  index=fx_cov_q.index, name="Qtrly std (%)")
std_a = pd.Series(np.sqrt(np.diag(fx_cov.values))  * 100,
                  index=fx_cov.index,  name="Annual std (%)")

comparison = pd.concat([std_q, std_a], axis=1)
comparison["Ratio (annual/qtrly)"] = comparison["Annual std (%)"] / comparison["Qtrly std (%)"]

print("Std check — ratio must equal √4 = 2.0 for all currencies:")
display(comparison.style.format("{:.4f}"))

print("\nAnnual covariance matrix (used in optimisation):")
fx_cov

---
## Section 2 -- Mean-Variance Efficient Frontier (foreign currencies only)

In [ ]:
# Strip 'KES_' prefix so the optimiser index is just the currency code
cov_df = fx_cov.rename(
    index=lambda x: x.split('_')[1],
    columns=lambda x: x.split('_')[1],
)
names = cov_df.index
n = len(names)

# Approximate sovereign borrowing costs (annual) from a KES perspective.
# Adjust to reflect current market conditions or your own assumptions.
interest_rates = {
    'USD': 0.043,
    'GBP': 0.045,
    'EUR': 0.025,
    'JPY': 0.005,
    'CNY': 0.025,
}

assumptions = pd.DataFrame(
    {
        'interest_rate':         [interest_rates.get(c, 0.03) for c in names],
        'expected_appreciation': [0.0] * n,
        'min_share':             [0.0] * n,
        'max_share':             [1.0] * n,
        'current_share':         [1.0 / n] * n,
    },
    index=names,
)
assumptions


In [ ]:
inputs = er.DebtFrontierInputs(
    cov_df=cov_df,
    assumptions=assumptions,
    name='kes_xlsx_basis',
    chartfolder='graph/',
)
inputs.widget()


---
## Section 3 -- Including Domestic (KES) Debt

Domestic KES debt carries **zero FX risk** but typically a higher
nominal interest rate. Adding it extends the frontier to lower-risk portfolios.

In [ ]:
# Extend covariance matrix with a KES (domestic) row/column of zeros -- no FX risk.
cov_df_dom = cov_df.reindex(
    index=list(cov_df.index) + ['KES'],
    columns=list(cov_df.columns) + ['KES'],
    fill_value=0.0,
)

# KES assumption row -- current_share=0 keeps foreign sum=1 in the widget indicator
kes_row = pd.DataFrame(
    {
        'interest_rate':         [0.12],   # ~12 % domestic rate -- placeholder
        'expected_appreciation': [0.0],
        'min_share':             [0.0],
        'max_share':             [1.0],
        'current_share':         [0.0],
    },
    index=['KES'],
)
assumptions_dom = pd.concat([assumptions, kes_row])

inputs_dom = er.DebtFrontierInputs(
    cov_df=cov_df_dom,
    assumptions=assumptions_dom,
    name='kes_basis_with_domestic',
    chartfolder='graph/',
)
inputs_dom.widget()


---
## Section 4 -- Scripted Scenario Loop (no widget)

For batch runs or report generation, use `inputs.plot()` directly.

In [ ]:
scenarios = {
    'dom_10pct': 0.10,
    'dom_12pct': 0.12,
    'dom_15pct': 0.15,
}

for scenario_name, dom_rate in scenarios.items():
    inp = er.DebtFrontierInputs(
        cov_df=cov_df_dom,
        assumptions=assumptions_dom.copy(),
        name=scenario_name,
        chartfolder='graph/',
    )
    inp.assumptions.at['KES', 'interest_rate'] = dom_rate
    print(f'Running scenario: {scenario_name}  (KES rate = {dom_rate:.0%})')
    inp.plot(export_formats=('svg',))
